In [34]:
"""
This script was developed to parallel process preformatted time series of input data needed for
the Kljun et. al 2d flux footprint prediction code and ultimately create monthly-ETo-weighted
footprint georeferenced footprint rasters. 

Checks are performed on the input data to handle data quality issues. The weighting method 
uses normalized hourly proportions of ASCE ETo computed from NLDAS v2 data for the closest cell.
NLDAS data is automatically downloaded using OpenDAP given Earthdata login info. Only months
with 20 days worth of good hourly data are used, specifically the criteria 13*20 = 260 or more hours
of data (only from hours between 6:00AM to 8:00 PM) must exist in a month. Checks are performed
to ensure final weighting procedure was successful at different steps of the process.

This script is not intended to be used by others but to document a workflow that was employed for
scientific purposes.
"""
import nldas_via_giovanni as nldas
import calc_footprint_FFP_climatology as ffp
import footprint_funcs as ff
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import refet
from pyproj import CRS, Transformer
import xarray
import requests
import multiprocessing as mp

__author__='John Volk'

In [35]:
# read metadata that has each sites' elevation used in ETr/ETo calcs
# AMF_meta_path = Path('master_flux_station_list.csv')
AMF_meta = pd.read_csv('stations_metadata.csv', index_col='SITE_ID')

In [47]:
# specify path with input CSV files for each station with 
# input time series of needed data, e.g. zm, u_star, L,...
in_dir = Path('input')
hourly_files = list(in_dir.glob('*.csv'))
print(hourly_files)

[PosixPath('input/US-Twt.csv')]


In [48]:
# Run the program!
#pool = mp.Pool(processes=8)
#pool.map(runner,hourly_files)
for hourly_file in hourly_files:
    runner(hourly_file)
print("ALL DONE!")

US-Twt coordinates: [-121.6531, 38.1087204]
US-Twt coordinates: (np.float64(-121.6531), np.float64(38.1087204))
original coordinates: 618082.0433162679 4218734.541993357
ry > 7.5
adjusted coordinates: 618075.0 4218735.0
Input data ranges from 2009-04-03 15:00:00 to 2017-04-04 09:00:00
Data is requested to NLDAS from 2009-04-03T13:00:00 to 2017-04-05T01:00:00
Requesting Rainf data
Request for Rainf data is done.
Requesting LWdown data
Request for LWdown data is done.
Requesting SWdown data
Request for SWdown data is done.
Requesting PotEvap data
Request for PotEvap data is done.
Requesting PSurf data
Request for PSurf data is done.
Requesting Qair data
Request for Qair data is done.
Requesting Tair data
Request for Tair data is done.
Requesting Wind_E data
Request for Wind_E data is done.
Requesting Wind_N data
Request for Wind_N data is done.
Date: 4/2009
actual hours: 268
Date: 2009/4/3, Hour: 15, Band: 1
Date: 2009/4/3, Hour: 16, Band: 2
Date: 2009/4/3, Hour: 17, Band: 3
Date: 2009/4

In [37]:
def read_compiled_input(path):
    """
    Check if required input data exists in file and is formatted appropriately.
    
    Input files should be hourly or finer temporal frequency, drops hours
    without required input data. 
    """
    ret = pd.DataFrame()
    need_vars = {'latitude','longitude','ET_corr','wind_dir','u_star','sigma_v','zm','hc','d','L'}
    str_value_columns = ['IGBP_land_classification','secondary_veg_type']

    #don't parse dates first check if required inputs exist to save processing time
    df=pd.read_csv(path, index_col='date', parse_dates=False)
    cols = df.columns
    check_1 = need_vars.issubset(cols)
    check_2 = len({'u_mean','z0'}.intersection(cols)) >= 1 # need one or the other
    
    # if either test failed then insufficient input data for footprint, abort
    if not check_1 or not check_2:
        print(f"ERROR: Required data is missing in the input file, {path}")
        return (ret, None, None)
    
    ret = df
    ret.index = pd.to_datetime(df.index)

    # make it hourly data
    # ret = ret.resample('h').mean()  # this can be used only when numeric columns exists in database
    agg_methods = {}
    for col_name in ret.columns:
        if col_name in str_value_columns:
            agg_methods[col_name] = 'first'
        else: 
            agg_methods[col_name] = 'mean'
    ret = ret.resample('h').agg(agg_methods)

    lat,lon = ret[['latitude','longitude']].values[0]
    keep_vars = need_vars.union({'u_mean','z0','IGBP_land_classification','secondary_veg_type'})
    drop_vars = list(set(cols).difference(keep_vars))
    ret.drop(drop_vars, axis=1, inplace=True)
    ret.dropna(subset=['wind_dir','u_star','sigma_v','d','zm','L','ET_corr'], how='any', inplace=True)
    # print(ret.head())
    
    return (ret, lat, lon)

In [38]:
def runner(path):
    """
    Given path to time series of site hourly (or finer) input data,
    compute daily ETo weighted footprint rasters. 
    
    Requires NASA Earthdata username and password to download NLDAS-v2
    primary forcing at point locations for estimated ASCE short ref. ET.
    """
    station = path.stem
    elevation = AMF_meta.loc[station, 'ELEVATION_METERS']
    utc_offset = AMF_meta.loc[station, 'UTC_OFFSET']
    if pd.isna(elevation):
        print(f"ERROR: Elevation is not provided in the metadata. Skipping {station}!")
        return
    if pd.isna(utc_offset):
        print(f"ERROR: UTC Offset is not provided in the metadata. Skipping {station}!")
        return
        
    df, latitude, longitude = read_compiled_input(path)
    if df.empty: 
        print(f"ERROR: Insufficient data exists. Skipping {station}!")
        return
    
    station_coord = (longitude, latitude)
    print(f"{station} coordinates: [{longitude}, {latitude}]")

    print(f"{station} coordinates: {station_coord}")
    # print(df.head())

    # get EPSG code from lat,long, convert to UTM
    EPSG=32700-np.round((45+latitude)/90.0)*100+np.round((183+longitude)/6.0)
    EPSG = int(EPSG)
    in_proj = CRS('EPSG:4326')
    out_proj = CRS('EPSG:{}'.format(EPSG))
    transformer = Transformer.from_crs(in_proj, out_proj, always_xy=True)
    (station_x,station_y) = transformer.transform(*station_coord)
    print('original coordinates:',station_x,station_y)
    # move coord to snap centroid to 30m grid, minimal distortion
    rx = station_x % 15
    if rx > 7.5:
        station_x += (15-rx)
        # final coords should be odd factors of 15
        if (station_x / 15) % 2 == 0:
            station_x -= 15
    else:    
        station_x -= rx
        if (station_x / 15) % 2 == 0:
            station_x += 15
    ry = station_y % 15
    if ry > 7.5:
        print('ry > 7.5')
        station_y += (15-ry )
        if (station_y / 15) % 2 == 0:
            station_y -= 15
    else:
        print('ry <= 7.5')
        station_y -= ry
        if (station_y / 15) % 2 == 0:
            station_y += 15
    print('adjusted coordinates:',station_x,station_y)

    #Other model parameters
    h_s = 2000. #Height of atmos. boundary layer [m] - assumed
    dx = 30. #Model resolution [m]
    origin_d = 300. #Model bounds distance from origin [m]
    #modify if needed
    start_hr = 5 # hours from 1 to 24
    end_hr = 17

    hours_array = np.arange(start_hr, end_hr+1)
    n_hrs = len(hours_array) 
    
    out_dir = Path('All_output')/'AMF_monthly'/f'{station}'

    if not out_dir.is_dir():
        out_dir.mkdir(parents=True, exist_ok=True)

    # get NLDAS data
    min_df_dt = df.index.min()
    max_df_dt = df.index.max()
    print(f"Input data ranges from {min_df_dt} to {max_df_dt}")
    # set to selected hours to make sure NLDAS data exists and adjust to UTC time
    start_dt_utc = min_df_dt.replace(hour=start_hr) - pd.Timedelta(hours=utc_offset) 
    end_dt_utc = max_df_dt.replace(hour=end_hr) - pd.Timedelta(hours=utc_offset)
    # if NLDAS data file exists, use it.
    nldas_ts_inf = out_dir/ f'nldas_ETr.csv'
    get_data = True
    if nldas_ts_inf.is_file():
        nldas_df = pd.read_csv(nldas_ts_inf, index_col='date', parse_dates=True).sort_index()
        if start_dt_utc >= nldas_df.index.min() and end_dt_utc <= nldas_df.index.max():
            print(f"Using the existing NLDAS file, {nldas_ts_inf}")
            get_data = False
    elif get_data:
        # get data from NLDAS and calculate ET
        start_dt_utc_str = start_dt_utc.strftime("%Y-%m-%dT%H:%M:%S")
        end_dt_utc_str = end_dt_utc.strftime("%Y-%m-%dT%H:%M:%S")
        nldas_df = get_nldas2_ETo(station_coord, start_dt_utc_str, end_dt_utc_str, elevation, out_dir)
        nldas_df = nldas_df.sort_index()
    # change UTC timestamp to station local timestamp
    nldas_df.index = nldas_df.index + pd.Timedelta(hours=utc_offset)
    # use only set hours
    nldas_df = nldas_df.between_time(f'{start_hr:02}:00', f'{end_hr:02}:00')

    df['date'] = df.index
    need_hrs = n_hrs * 20 # 20 days worth of hours (6-8)
    months = [g for n, g in df.groupby(pd.Grouper(key='date',freq='ME'))]

    #Loop through monthly grouped slices
    for mdf in months:
        if mdf.empty:
            print(f'No data for {month_str}/{year_str}, skipping.')
            continue
            
        #Subset dataframe to only values in day of year
        year_str = mdf.index.year[0]
        month_str = mdf.index.month[0]
        print(f'Date: {month_str}/{year_str}')
        
        temp_df=mdf.between_time(f'{start_hr:02}:00', f'{end_hr:02}:00')
        actual_hrs = len(temp_df)
        print(f"actual hours: {actual_hrs}")

        # check on n hours per day
        if actual_hrs < need_hrs:
            print(f'Less than {need_hrs} hours of data for {month_str}/{year_str}, skipping.')
            continue

        new_dat = None

        out_f = out_dir/ f'{year_str}-{month_str:02}.tif'

        final_outf = out_dir/f'{year_str}-{month_str:02}_weighted.tif'

        # make hourly band raster for the day
        band=1
        band_date_dict = []
        for date, temp_line in temp_df.iterrows():
            hour = date.hour
            print(f'Date: {year_str}/{month_str}/{date.day}, Hour: {hour}, Band: {band}')

            try:
                if temp_line.empty: 
                    print(f'Missing all data for {date,hour} skipping')
                    temp_ffp = None
                    band_date_dict.append({'band': band, 'date':date})
                    band+=1
                    continue

                zm = temp_line.zm - temp_line.d
                z0 = np.array(temp_line.z0) if 'z0' in temp_line else None
                u_mean = np.array(temp_line.u_mean) if 'u_mean' in temp_line else None
                if u_mean is not None: z0 = None
                
                #Calculate footprint
                temp_ffp = ffp.ffp_climatology(domain=[-origin_d,origin_d,-origin_d,origin_d],dx=dx,dy=dx,
                                        zm=np.array(zm), h=np.array(h_s), rs=None, 
                                        z0=z0, ol=np.array(temp_line['L']),
                                        sigmav=np.array(temp_line['sigma_v']),
                                        ustar=np.array(temp_line['u_star']), 
                                        umean=u_mean,
                                        wind_dir=np.array(temp_line['wind_dir']),
                                        crop=0,fig=0,verbosity=0)
                ####verbosoity=2 prints out errors; if z0 triggers errors, use umean

                f_2d = np.array(temp_ffp['fclim_2d'])
                x_2d = np.array(temp_ffp['x_2d']) + station_x
                y_2d = np.array(temp_ffp['y_2d']) + station_y
                f_2d = f_2d*dx**2
                
                #Calculate affine transform for given x_2d and y_2d
                affine_transform = ff.find_transform(x_2d,y_2d)

                #Create data file if not already created
                if new_dat is None:
                    #print(f_2d.shape)
                    new_dat = rasterio.open(
                        out_f,'w',driver='GTiff',dtype=rasterio.float64,
                        count=actual_hrs,height=f_2d.shape[0],width=f_2d.shape[1],
                        transform=affine_transform, crs=out_proj.srs,
                        nodata=0.00000000e+000
                    )

            except Exception as e:
                print(f'Hour {hour} footprint failed, band {band} not written.')
                print(f'Exception: {e}')

                temp_ffp = None
                band_date_dict.append({'band': band, 'date':date})
                band+=1
                continue

            #Mask out points that are below a % threshold (defaults to 90%)
            f_2d = ff.mask_fp_cutoff(f_2d)

            #Write the new band
            new_dat.write(f_2d, band)

            #Update tags with metadata
            tag_dict = {'hour':f'{hour*100:04}',
                        'wind_dir':np.array(temp_line['wind_dir']),
                        'total_footprint':np.nansum(f_2d)}

            new_dat.update_tags(band,**tag_dict)
            band_date_dict.append({'band': band, 'date':date})            
            band+=1

        #Close dataset if it exists
        try:
            new_dat.close()
        except:
            print(f'ERROR: could not write footprint for site: {station}:\nto: {out_f}')
            continue # skip to next month...
            
        band_date_df = pd.DataFrame(band_date_dict)
        band_count = len(band_date_df)
        print(f"size check, actual hrs - {actual_hrs} vs band countvs - {band_count}")
        # do hourly weighting - do not necessarily need to do this all in the same loop
        src = rasterio.open(out_f)
        # hourly fetch scalar sums and normalized fetch rasters
        global_sum = np.zeros(shape=(band_count))
        normed_fetch_rasters = [] 
        for index, row in band_date_df.iterrows():
            arr = src.read(row.band)
            global_sum[index] = arr.sum()
            if global_sum[index] == 0:
                tmp = np.zeros_like(arr)
            else:
                tmp = arr/global_sum[index]
            normed_fetch_rasters.append(tmp)

        
        # get NLDAS ts calc fraction of daily ETo
        nldas_sub_df = nldas_df.loc[(nldas_df.index.year==year_str)&(nldas_df.index.month==month_str)].copy()
        # ETo = nldas_df.loc[(nldas_df.index.year==year_str)&(nldas_df.index.month==month_str),'ETo']
        # deal with negative ETo value proportions
        nldas_sub_df['min_max_normed_ETo'] =\
            (nldas_sub_df['ETo']-min(nldas_sub_df['ETo']))/(max(nldas_sub_df['ETo'])-min(nldas_sub_df['ETo']))

        #min_max_normed_ETo = (ETo-min(ETo))/(max(ETo)-min(ETo)) # deal with negative ETo value proportions
        # take out hours where footprint does not exist
        exist_mask = nldas_sub_df.index.isin(band_date_df['date'])
        indices_to_zero = nldas_sub_df.index[~exist_mask]
        # print(f"indices to set ETo = 0: {indices_to_zero}") 
        nldas_sub_df.loc[indices_to_zero, 'min_max_normed_ETo'] = 0
        # print(nldas_sub_df['min_max_normed_ETo'])
        # print(nldas_sub_df['min_max_normed_ETo'].sum())
            
        # after removing hours now calculate hourly proportions
        nldas_df.loc[
            (nldas_df.index.year == year_str) & (nldas_df.index.month == month_str), 
            'ETo_hr_props'
        ] = nldas_sub_df['min_max_normed_ETo'] / nldas_sub_df['min_max_normed_ETo'].sum()
        

        # weight normed hourly fetch rasters by hourly ETo proportions
        month_slice = nldas_df.loc[
            (nldas_df.index.year == year_str) & (nldas_df.index.month == month_str)
        ]
        for index, row in band_date_df.iterrows():
            normed_fetch_rasters[index] =\
                normed_fetch_rasters[index]*month_slice.loc[row.date, 'ETo_hr_props']

        # Last calculation, sum the weighted hourly rasters to a single monthly fetch raster
        final_footprint = sum(normed_fetch_rasters)
        # assert np.isclose(final_footprint.sum(), 1), f'check 1 failed! {final_footprint.sum()}\n'
        if not np.isclose(final_footprint.sum(), 1):
            print(f'check 1 failed! {final_footprint.sum()}\n')
            continue
        # next check
        for index, raster in enumerate(normed_fetch_rasters):
            date_index = band_date_df.loc[index,'date']
            if not np.isclose(
                month_slice.loc[date_index, 'ETo_hr_props'], raster.sum()
            ):
                print(f'check 2 failed for {date_index}!!!')
                continue
            """
            assert np.isclose(
                month_slice.iloc[hour, prop_col_indx], raster.sum()
            ), f'check 2 failed for hour {hour}!!!'
            """

        # finally, write daily corrected raster with UTM zone reference 
        corr_raster_path = final_outf
        out_raster = rasterio.open(
            corr_raster_path,'w',driver='GTiff',dtype=rasterio.float64,
            count=1,height=final_footprint.shape[0],width=final_footprint.shape[1],
            transform=src.transform, crs=out_proj.srs, nodata=0.00000000e+000
        )
        out_raster.write(final_footprint,1)
        out_raster.close()

    if 'ETo_hr_props' in nldas_df:
        nldas_ts_outf = out_dir/ f'nldas_ETr_props.csv'
        nldas_df.round(4).to_csv(nldas_ts_outf)

    print(f"{station} is done!")

        

In [39]:
def get_nldas2_ETo(
    coords: tuple[float, float], 
    start_dt_utc: str,  # Date must be the format of YYYY-MM-DDThh:mm:ss in UTC
    end_dt_utc: str,
    elevation: float,
    out_dir: str,
    api_token=""      # api_token can be empty if you have three files, .netrc, .dodsrc and .urs_cookies, created in the root direcotry
):
    zm = 10  # nldas2 windspeed height is 10 m  <- Used in refET calculation 
    # Date must be the format of YYYY-MM-DDThh:mm:ss in UTC
    # start_date = start_date.strftime("%Y-%m-%dT%H:%M:%S")
    # end_date = end_date.strftime("%Y-%m-%dT%H:%M:%S")
    print(f"Data is requested to NLDAS from {start_dt_utc} to {end_dt_utc}")

    # api request information    
    data = "NLDAS_FORA0125_H_2_0"
    variables = ["Rainf", "LWdown", "SWdown", "PotEvap", "PSurf", "Qair", "Tair", "Wind_E", "Wind_N"]
    """
    # variable infomation for NLDAS2
    variable_names = ["Precipitation hourly total", "Surface DW longwave radiation flux", "Surface DW shortwave radiation flux", 
                      "Potential evaporation", "Surface pressure", "2-m above ground specific humidity", "2-m above ground temperature",
                      "10-m above ground zonal wind", "10-m above ground meridional wind"]
    variable_units = ["kg/m2", "W/m2", "W/m2", "kg/m2", "Pa", "kg/kg", "K", "m/s", "m/s"]
    """

    nldas_df = nldas.get_nldas2_data(coords[1], coords[0], start_dt_utc, end_dt_utc, data, variables, "", api_token)
    nldas_df.index.name = 'date'
    # print(nldas_df.head())

    
    nldas_df['pair'] = nldas_df['PSurf'] / 1000 # nldas air pres in Pa convert to kPa
    # sph = ds.get('SPF_H_110_HTGL').data # kg/kg
    nldas_df['ea'] = refet.calcs._actual_vapor_pressure(q=nldas_df['Qair'].to_numpy(), pair=nldas_df['pair'].to_numpy())  # ea in kPa
    # calculate hourly wind
    nldas_df['wind'] = np.sqrt(nldas_df['Wind_E'] ** 2 + nldas_df['Wind_N'] ** 2)
    # get temp convert to C
    nldas_df['temp'] = nldas_df['Tair'] - 273.15
    # get rs
    unit_dict = {'rs': 'w/m2'}

    nldas_df['doy'] = nldas_df.index.dayofyear
    nldas_df['HH'] = nldas_df.index.hour
    
    # create refet object for calculating
    for index, row in enumerate(nldas_df.itertuples()):

        etr_calculator = refet.Hourly(
            tmean=row.temp,
            ea=row.ea,
            rs=row.SWdown,
            uz=row.wind,
            zw=zm,
            elev=elevation,
            lat=coords[1],
            lon=coords[0],
            doy=row.doy,
            time=row.HH,
            method='asce',
            input_units=unit_dict
        )  # HH must be int
        # Calculate the ETr and store it in the DataFrame
        nldas_df.loc[row.Index, 'ETr'] = etr_calculator.etr()[0]
        nldas_df.loc[row.Index, 'ETo'] = etr_calculator.eto()[0]
    
    # Save     
    ETr_df = pd.DataFrame(columns=['ETr','ETo','ea','sph','wind','pair','temp','rs'])
    ETr_df['ETr'] = nldas_df['ETr']
    ETr_df['ETo'] = nldas_df['ETo']
    ETr_df['ea'] = nldas_df['ea']
    ETr_df['sph'] = nldas_df['Qair']
    ETr_df['wind'] = nldas_df['wind']
    ETr_df['pair'] = nldas_df['pair']
    ETr_df['temp'] = nldas_df['temp']
    ETr_df['rs'] = nldas_df['SWdown']
    ETr_df.index.name = 'date'
    # print(ETr_df.head())
    
    # save data
    nldas_ts_outf = out_dir/ f'nldas_ETr.csv'
    ETr_df.round(4).to_csv(nldas_ts_outf)
    
    return ETr_df
       